### 重试机制

Agent 循环里的工具调用、模型调用，全都是**外部依赖**：HTTP 接口、数据库、向量库、第三方 LLM。而外部调用失败是常态——网络抖动、上游 5xx、限流 429、连接被重置、响应慢到超时。LangGraph 里节点一旦抛异常，默认会让**整张图失败**。

所以生产环境里的 Agent 必须配齐「失败处理」三件套：

| 手段 | 作用 | 配置位置 |
| --- | --- | --- |
| **重试**（本节重点） | 偶发故障自己扛过去，不打扰用户 | 节点级 `retry_policy` |
| **超时** | 把卡死的调用及时打断，不让它一直占着资源 | 节点级 `timeout` |
| **降级兜底** | 重试也失败时给一个可接受的答案，而不是整图报错 | 节点级 `error_handler` |

> 本节所有行为都在 `langgraph==1.2.11` 下实测过；`RetryPolicy` 从 0.2.24 引入，中间版本默认值有过变化，以你环境里 `RetryPolicy()` 的实际字段为准。


#### 1. 最小示例：重试用尽后，异常才往上抛

先看最直接的用法：在 `add_node` 上挂一个 `RetryPolicy`。`max_attempts=3` 的含义是**总共执行 3 次**（含第一次），三次都失败才把异常抛给调用方。

注意：如果不写 `try/except`，这个 cell 会以 `HTTPError` 报错结束——因为「重试耗尽」在 LangGraph 看来就是一次普通的节点失败。


In [1]:
from langchain_core.messages import HumanMessage
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langgraph.types import RetryPolicy
from loguru import logger
from typing import TypedDict

from requests import HTTPError


class EmptyState(TypedDict):
    pass


attempts = {"count": 0}   # 闭包计数器：用来观察节点到底被重试了几次


def node_a(state: EmptyState) -> EmptyState:
    attempts["count"] += 1
    logger.info(f"node_a 第 {attempts['count']} 次执行")
    raise HTTPError("模拟接口返回 500")


# 构建图：retry_policy 挂在节点上，max_attempts=3 表示「总共执行 3 次」（含第一次）
builder = StateGraph(EmptyState)
builder.add_node("node_a", node_a, retry_policy=RetryPolicy(max_attempts=3))
builder.add_edge(START, "node_a")
builder.add_edge("node_a", END)

graph = builder.compile()
try:
    print(graph.invoke({"messages": [HumanMessage(content="你好")]}))
except HTTPError as e:
    # 3 次都失败后异常才会向上抛出；不兜住的话这个 cell 会以报错结束
    print(f"重试 {attempts['count']} 次后仍然失败：{type(e).__name__}: {e}")


2026-09-27 15:06:29.349 | INFO     | __main__:node_a:20 - node_a 第 1 次执行


2026-09-27 15:06:30.209 | INFO     | __main__:node_a:20 - node_a 第 2 次执行


2026-09-27 15:06:32.194 | INFO     | __main__:node_a:20 - node_a 第 3 次执行


重试 3 次后仍然失败：HTTPError: 模拟接口返回 500


#### 2. `RetryPolicy` 的参数与语义

`RetryPolicy` 是 `langgraph.types` 里的 `NamedTuple`，所有字段都有默认值：

| 字段 | 默认值 | 说明 |
| --- | --- | --- |
| `max_attempts` | `3` | 总执行次数（**含第一次**），用完才把异常抛出去 |
| `initial_interval` | `0.5` | 第一次重试前等待的秒数 |
| `backoff_factor` | `2.0` | 退避倍数（指数退避） |
| `max_interval` | `128.0` | 单次等待上限 |
| `jitter` | `True` | 是否加 `0~1s` 随机抖动，避免一批请求同时重试 |
| `retry_on` | `default_retry_on` | 哪些异常触发重试：异常类 / 类列表 / 判定函数 |

核心逻辑在 `langgraph/pregel/_retry.py` 的 `run_with_retry` 里，三件事值得记住：

```python
# 1) 每次重试前都会清空该节点已写入的 state，失败尝试不会留下脏数据
task.writes.clear()

# 2) 退避间隔 = min(max_interval, initial_interval * backoff_factor ** (失败次数 - 1)) (+ jitter)
interval = min(policy.max_interval, policy.initial_interval * (policy.backoff_factor ** (attempts - 1)))
sleep_time = interval + random.uniform(0, 1) if policy.jitter else interval

# 3) 每次重试都会打一条日志（stdlib logging，logger 名是 langgraph.pregel._retry）
logger.info(f"Retrying task {task.name} after {sleep_time:.2f} seconds (attempt {attempts}) after {exc.__class__.__name__} {exc}", exc_info=exc)
```

所以**等待总时长**是可以估出来的：默认参数下 `max_attempts=5` 大约要空转 `0.5 + 1 + 2 + 4 ≈ 7.5s`（还不含 jitter）。调试时记得把 `initial_interval` 调小，否则每次跑 cell 都要等好几秒。

另外两点：

- **策略只能挂在节点上**。1.2.11 的 `compile()` 已经不接受 `retry_policy` 了（`add_node` / `set_node_defaults` 才支持），而 `langgraph.constants` 那些老写法在 1.x 里也早就搬到 `langgraph.graph` / `langgraph.types` 了。
- **中断不会被重试**。`GraphBubbleUp`（如 `interrupt_before` 触发的人工审批、`NodeInterrupt`）在重试逻辑里被显式放行抛出，不会被当成失败重试。
- **重试耗尽抛出的异常带上下文注释**（`Exception.add_note`），排查时能直接看出是哪个节点的哪次任务挂了：

```python
try:
    graph.invoke({"messages": [HumanMessage(content="你好")]})
except HTTPError as e:
    print("".join(getattr(e, "__notes__", [])))
# During task with name 'node_a' and id '60d570ae-2ff3-b670-6ae4-64ae1044c3ac'
```


##### 例 1：偶发失败重试到成功，节点内部还能知道"这是第几次"

比"一直失败"更常见的是"抖一下就好了"。1.2.11 会把尝试次数注入 `runtime.execution_info.node_attempt`，节点内部可以直接读——用来做"第 2 次才换降级参数"这类逻辑。


In [2]:
import time
from operator import add
from typing import Annotated

import requests
from langgraph.graph import StateGraph
from langgraph.runtime import get_runtime
from langgraph.types import RetryPolicy


class RetryState(TypedDict):
    """带一个可累加的 logs，方便观察「谁执行过、成功几次」。"""

    logs: Annotated[list[str], add]


attempts = {"n": 0}   # 用可变字典当计数器（Python 里不能给全局变量 +=）


def flaky_node(state: RetryState) -> RetryState:
    attempts["n"] += 1
    # 1.2.11 把「第几次尝试」注入 runtime，节点内部可以直接感知
    node_attempt = get_runtime().execution_info.node_attempt
    print(f"  第 {attempts['n']} 次执行，node_attempt = {node_attempt}")
    if attempts["n"] < 3:
        raise requests.HTTPError("503 服务暂时不可用")
    return {"logs": [f"第 {attempts['n']} 次成功"]}


flaky_builder = StateGraph(RetryState)
flaky_builder.add_node(
    "flaky_node",
    flaky_node,
    # 调试时把 initial_interval 调小，否则默认要等 0.5s + 1s 才成功
    retry_policy=RetryPolicy(max_attempts=5, initial_interval=0.01),
)
flaky_builder.add_edge(START, "flaky_node")
flaky_builder.add_edge("flaky_node", END)

flaky_graph = flaky_builder.compile()
print("最终 state =", flaky_graph.invoke({"logs": []}))


  第 1 次执行，node_attempt = 1


  第 2 次执行，node_attempt = 2
  第 3 次执行，node_attempt = 3
最终 state = {'logs': ['第 3 次成功']}


##### 例 2：指数退避 + jitter，实测一遍

用 `jitter=False` 关掉随机抖动，就能干净地看出 `0.2 → 0.4 → 0.8` 的等比增长；生产环境则应该**保留** `jitter=True`，否则一批并发请求会卡在同一个时刻一起重试（惊群），把刚恢复的服务再打垮。


In [3]:
stamps: list[float] = []


def always_unavailable(state: RetryState) -> RetryState:
    stamps.append(time.perf_counter())
    # 这里用 builtin 的 ConnectionError：默认 retry_on 只认它（requests 那个不认，见第 3 节）
    raise ConnectionError("上游一直连不上")


backoff_builder = StateGraph(RetryState)
backoff_builder.add_node(
    "always_unavailable",
    always_unavailable,
    retry_policy=RetryPolicy(
        max_attempts=4,
        initial_interval=0.2,
        backoff_factor=2.0,
        max_interval=10.0,
        jitter=False,     # 关掉抖动，方便观察等比退避
    ),
)
backoff_builder.add_edge(START, "always_unavailable")
backoff_builder.add_edge("always_unavailable", END)

try:
    backoff_builder.compile().invoke({"logs": []})
except ConnectionError as e:
    print("重试耗尽，异常上抛 ->", type(e).__name__, e)

gaps = [round(stamps[i + 1] - stamps[i], 3) for i in range(len(stamps) - 1)]
print(f"实际执行 {len(stamps)} 次，相邻两次间隔 = {gaps}")
print("对照公式 initial_interval * backoff_factor ** (失败次数 - 1)：0.2 / 0.4 / 0.8")


重试耗尽，异常上抛 -> ConnectionError 上游一直连不上
实际执行 4 次，相邻两次间隔 = [0.205, 0.403, 0.801]
对照公式 initial_interval * backoff_factor ** (失败次数 - 1)：0.2 / 0.4 / 0.8


#### 3. `retry_on`：哪些异常值得重试

默认策略 `default_retry_on` 的思路是「**外部故障重试，代码 bug 不重试**」。1.2.11 里的实际判定如下（下面这张表是逐个跑出来的，不是照抄文档）：

| 异常 | 默认是否重试 | 说明 |
| --- | --- | --- |
| builtin `ConnectionError` | ✅ | 明确写在白名单里 |
| `requests.ConnectionError` / `requests.Timeout` / `ReadTimeout` | ❌ | **坑**：requests 的异常继承自 `OSError`，命中"不重试"名单 |
| `requests.HTTPError`（404 / 503 / 没带 response） | ✅ | 坑：`if exc.response` 走的是 `Response.__bool__`，而错误响应是 falsy，所以一律重试 |
| `httpx.ConnectError` / `httpx.ReadTimeout` | ✅ | httpx 的异常不继承 `OSError` |
| `httpx.HTTPStatusError` 404 / 503 | ❌ / ✅ | httpx 这条是真按状态码判断的 |
| builtin `TimeoutError` | ❌ | 它是 `OSError` 子类 |
| `NodeTimeoutError` | ✅ | 框架**故意**不让它继承 `TimeoutError`，就是为了能被默认策略重试 |
| `ValueError` / `TypeError` / `KeyError` / `RuntimeError` / `JSONDecodeError` … | ❌ | 代码写错了，重试只会浪费时间 |
| 其它异常 | ✅ | 兜底放行 |

结论：**默认策略对 `requests` 的连接类异常是"不重试"的**，如果你用 `requests` 调外部接口，务必显式写 `retry_on`。下面把这个差异跑出来，再演示两种自定义写法。


In [4]:
import unicodedata

import httpx


def pad(text: str, width: int = 44) -> str:
    """中文按 2 个字符宽度补齐，让表格对得上。"""
    used = sum(2 if unicodedata.east_asian_width(ch) in "WF" else 1 for ch in text)
    return text + " " * max(0, width - used)


def probe(exc_factory, label: str, **policy_kwargs) -> int:
    """跑一个「一上来就抛异常」的节点，返回它实际被执行了几次（1 = 没重试）。"""
    counter = {"n": 0}

    def node(state: RetryState) -> RetryState:
        counter["n"] += 1
        raise exc_factory()

    builder = StateGraph(RetryState)
    builder.add_node(
        "x",
        node,
        retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.001, **policy_kwargs),
    )
    builder.add_edge(START, "x")
    builder.add_edge("x", END)
    try:
        builder.compile().invoke({"logs": []})
    except Exception:
        pass
    print(f"  {pad(label)}执行 {counter['n']} 次")
    return counter["n"]


def http_error(status: int) -> requests.HTTPError:
    response = requests.Response()
    response.status_code = status
    return requests.HTTPError(f"{status} 服务端错误", response=response)


def httpx_status_error(status: int) -> httpx.HTTPStatusError:
    request = httpx.Request("GET", "https://example.com/api")
    return httpx.HTTPStatusError(f"{status} 服务端错误", request=request, response=httpx.Response(status, request=request))


print("【默认 retry_on】")
probe(lambda: ValueError("参数写错了"), "ValueError（代码 bug）")
probe(lambda: KeyError("字段不存在"), "KeyError（代码 bug）")
probe(lambda: ConnectionError("断网"), "builtin ConnectionError")
probe(lambda: requests.ConnectionError("断网"), "requests.ConnectionError")
probe(lambda: requests.ReadTimeout("读超时"), "requests.ReadTimeout")
probe(lambda: http_error(404), "requests.HTTPError 404")
probe(lambda: httpx_status_error(404), "httpx.HTTPStatusError 404")
probe(lambda: httpx_status_error(503), "httpx.HTTPStatusError 503")
probe(lambda: httpx.ConnectError("连接失败"), "httpx.ConnectError")

print("【自定义 retry_on：异常类列表】")
probe(lambda: requests.ConnectionError("断网"), "requests.ConnectionError 显式加入白名单",
      retry_on=[requests.ConnectionError, requests.ReadTimeout])

print("【自定义 retry_on：判定函数】")
probe(lambda: requests.HTTPError("429 Too Many Requests"), "只重试 429 限流",
      retry_on=lambda e: "429" in str(e))
probe(lambda: requests.HTTPError("404 Not Found"), "404 不在判定函数里 -> 不重试",
      retry_on=lambda e: "429" in str(e))


【默认 retry_on】
  ValueError（代码 bug）                      执行 1 次
  KeyError（代码 bug）                        执行 1 次


  builtin ConnectionError                     执行 3 次
  requests.ConnectionError                    执行 1 次
  requests.ReadTimeout                        执行 1 次


  requests.HTTPError 404                      执行 3 次
  httpx.HTTPStatusError 404                   执行 1 次


  httpx.HTTPStatusError 503                   执行 3 次


  httpx.ConnectError                          执行 3 次
【自定义 retry_on：异常类列表】


  requests.ConnectionError 显式加入白名单     执行 3 次
【自定义 retry_on：判定函数】


  只重试 429 限流                             执行 3 次
  404 不在判定函数里 -> 不重试                执行 1 次


1

#### 4. 多套策略：`retry_policy` 也可以传一个列表

`retry_policy` 的类型是 `RetryPolicy | Sequence[RetryPolicy]`，多个策略时 LangGraph 会**按顺序找第一个 `retry_on` 命中的策略**来决定重试次数和退避参数（`run_with_retry` 里的 `for policy in retry_policy: if _should_retry_on(policy, exc): break`）。

由此得到两条实践规则：

- 兜底策略要放最后，并且**显式收窄**它的 `retry_on`；否则一个宽松的默认策略排在前面会把后面的精细策略"抢答"掉。
- 每套策略自己声明 `max_attempts` 和退避参数，互不影响。


In [5]:
def probe_policies(exc_factory, label: str, policies) -> int:
    counter = {"n": 0}

    def node(state: RetryState) -> RetryState:
        counter["n"] += 1
        raise exc_factory()

    builder = StateGraph(RetryState)
    builder.add_node("x", node, retry_policy=policies)
    builder.add_edge(START, "x")
    builder.add_edge("x", END)
    try:
        builder.compile().invoke({"logs": []})
    except Exception:
        pass
    print(f"  {pad(label, 60)}执行 {counter['n']} 次")
    return counter["n"]


# 专用策略：网络类异常多试几次（注意这里用的是 builtin ConnectionError）
network_policy = RetryPolicy(max_attempts=4, initial_interval=0.001, retry_on=ConnectionError)
# 宽松兜底：用默认 retry_on（几乎什么都重试），但只允许试 1 次
loose_fallback = RetryPolicy(max_attempts=1, initial_interval=0.001)
# 窄兜底：显式收窄，谁也不匹配
narrow_fallback = RetryPolicy(max_attempts=1, initial_interval=0.001, retry_on=lambda e: False)

print("顺序不同，结果完全不同：")
probe_policies(lambda: ConnectionError("断网"),
               "[网络策略(4次), 宽松兜底(1次)] -> 命中网络策略", [network_policy, loose_fallback])
probe_policies(lambda: ConnectionError("断网"),
               "[宽松兜底(1次) 在前, 网络策略(4次)] -> 被抢答，只 1 次",
               [loose_fallback, network_policy])

print("把兜底策略的 retry_on 收窄就稳定了：")
probe_policies(lambda: ConnectionError("断网"),
               "[网络策略(4次), 窄兜底(1次, 永不命中)]", [network_policy, narrow_fallback])
probe_policies(lambda: ValueError("代码 bug"),
               "[网络策略, 窄兜底] 遇到 ValueError -> 不重试", [network_policy, narrow_fallback])
probe_policies(lambda: requests.ConnectionError("断网"),
               "[网络策略只认 builtin] -> 两条都不匹配，不重试", [network_policy, narrow_fallback])


顺序不同，结果完全不同：


  [网络策略(4次), 宽松兜底(1次)] -> 命中网络策略              执行 4 次
  [宽松兜底(1次) 在前, 网络策略(4次)] -> 被抢答，只 1 次      执行 1 次
把兜底策略的 retry_on 收窄就稳定了：


  [网络策略(4次), 窄兜底(1次, 永不命中)]                      执行 4 次
  [网络策略, 窄兜底] 遇到 ValueError -> 不重试                执行 1 次
  [网络策略只认 builtin] -> 两条都不匹配，不重试              执行 1 次


1

#### 5. 重试与 state、步数的关系

三个容易搞混的点，都跟"重试到底算不算一步"有关：

1. **重试发生在同一个节点任务内部，不额外消耗 super-step**。上一个 cell 里 `flaky_node` 试了 3 次，但整张图只走了 2 步（`flaky_node` + `after`）。所以重试和上一节讲的 `recursion_limit` 是两套独立的预算：重试再多也不会触发 `GraphRecursionError`。
2. **失败尝试写入的 state 会被丢弃**（每次尝试前 `task.writes.clear()`），所以只有最后一次成功（或最终失败前的那次）结果会进 state，不会出现"重试 3 次、日志里就有 3 条重复记录"的情况。
3. **`stream_mode="debug"` 看不到重试**——它只按 task/task_result 打点，一次节点执行就是一条记录。要观察重试，请打开 `langgraph.pregel._retry` 这个 logger（默认 WARNING 级别，所以平时是静默的）：

```python
import logging

retry_logger = logging.getLogger("langgraph.pregel._retry")
retry_logger.setLevel(logging.INFO)
retry_logger.addHandler(logging.StreamHandler())   # 日志行长这样：
# Retrying task always_unavailable after 0.20 seconds (attempt 1) after ConnectionError 上游一直连不上
```

注意这条记录带了 `exc_info`，所以终端里会附带原始异常 traceback；生产环境建议把它接进结构化日志（LangSmith / OpenTelemetry）并按 `node` + `attempt` 做聚合告警。


In [6]:
from langgraph.checkpoint.memory import InMemorySaver

attempts["n"] = 0   # 复用例 1 的 flaky_node

steps_builder = StateGraph(RetryState)
steps_builder.add_node("flaky_node", flaky_node,
                       retry_policy=RetryPolicy(max_attempts=5, initial_interval=0.001))
steps_builder.add_node("after", lambda state: {"logs": ["after 已执行"]})
steps_builder.add_edge(START, "flaky_node")
steps_builder.add_edge("flaky_node", "after")
steps_builder.add_edge("after", END)
steps_graph = steps_builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "retry-steps"}}
for chunk in steps_graph.stream({"logs": []}, config, stream_mode="updates"):
    print("  update chunk:", list(chunk.keys()))

snapshot = steps_graph.get_state(config)
print("总步数 =", snapshot.metadata["step"], "（flaky_node 重试到第 3 次才成功，也只占 1 步）")
print("state.logs =", snapshot.values["logs"], "（失败的那两次没有留下任何脏数据）")

print("--- stream_mode='debug'：重试不会产生额外 task 事件 ---")
attempts["n"] = 0
for record in steps_graph.stream({"logs": []}, {"configurable": {"thread_id": "retry-debug"}},
                                 stream_mode="debug"):
    payload = record["payload"]
    print(f"  step={record['step']} {record['type']:<12} node={payload.get('name', '')} error={payload.get('error')}")


  第 1 次执行，node_attempt = 1
  第 2 次执行，node_attempt = 2
  第 3 次执行，node_attempt = 3
  update chunk: ['flaky_node']
  update chunk: ['after']
总步数 = 2 （flaky_node 重试到第 3 次才成功，也只占 1 步）
state.logs = ['第 3 次成功', 'after 已执行'] （失败的那两次没有留下任何脏数据）
--- stream_mode='debug'：重试不会产生额外 task 事件 ---
  step=-1 checkpoint   node= error=None
  step=0 checkpoint   node= error=None
  step=1 task         node=flaky_node error=None
  第 1 次执行，node_attempt = 1


  第 2 次执行，node_attempt = 2
  第 3 次执行，node_attempt = 3
  step=1 task_result  node=flaky_node error=None
  step=1 checkpoint   node= error=None
  step=2 task         node=after error=None
  step=2 task_result  node=after error=None
  step=2 checkpoint   node= error=None


#### 6. 重试耗尽之后：`error_handler` 降级

重试解决的是"偶发故障"，解决不了"服务真的挂了"。这时有两种选择：让异常冒到调用方（由上层兜底），或者在节点上挂一个 `error_handler`，把失败变成一条正常的降级结果。

1.2.11 的用法是给节点加 `error_handler=`，并在 handler 的签名里声明一个 `NodeError` 参数，框架会把失败上下文注入进来：

```python
def handler(state: State, error: NodeError) -> Command:
    # error.node  = 失败的节点名，error.error = 原始异常
    return Command(update={"logs": [...]})     # 返回 Command 就等于"图正常跑完了"
```

这样整张图不会失败，可以继续往下走分支。几个细节：

- handler 节点**不会再被自己触发**（避免 handler 失败 → 再调 handler 的死循环）。
- `set_node_defaults(retry_policy=...)` 设的默认重试**也会应用到 handler 节点**上，所以 handler 里的网络调用同样值得加重试。
- handler 里不要顺手去写外部状态（写了一半失败的副作用还在），它的定位是"给出一个可接受的答案"。


In [7]:
from langgraph.errors import NodeError
from langgraph.types import Command


def broken_node(state: RetryState) -> RetryState:
    raise requests.HTTPError("503 一直坏")


def fallback_handler(state: RetryState, error: NodeError) -> Command:
    """重试耗尽后由框架调用：把失败信息包成 NodeError 注入进来。"""
    print(f"  降级处理：失败节点 = {error.node}，异常 = {type(error.error).__name__}: {error.error}")
    return Command(update={"logs": [f"{error.node} 连续失败，已降级返回"]})


handler_builder = StateGraph(RetryState)
handler_builder.add_node(
    "broken_node",
    broken_node,
    retry_policy=RetryPolicy(max_attempts=2, initial_interval=0.001),
    error_handler=fallback_handler,          # 重试也失败 -> 走降级
)
handler_builder.add_edge(START, "broken_node")
handler_builder.add_edge("broken_node", END)
print("带 error_handler：", handler_builder.compile().invoke({"logs": []}))

# 对照组：同一个节点不配 error_handler，异常一路冒到 invoke
plain_builder = StateGraph(RetryState)
plain_builder.add_node("broken_node", broken_node,
                       retry_policy=RetryPolicy(max_attempts=2, initial_interval=0.001))
plain_builder.add_edge(START, "broken_node")
plain_builder.add_edge("broken_node", END)
try:
    plain_builder.compile().invoke({"logs": []})
except requests.HTTPError as e:
    print("不配 error_handler：整图失败 ->", type(e).__name__, e)


  降级处理：失败节点 = broken_node，异常 = HTTPError: 503 一直坏
带 error_handler： {'logs': ['broken_node 连续失败，已降级返回']}


不配 error_handler：整图失败 -> HTTPError 503 一直坏


#### 7. 超时：重试之前先把"卡住"变失败

没有超时的重试是危险的：第一次调用卡了 60 秒，你以为 0.5 秒后就重试，其实它在原地挂着。1.2.11 用 `TimeoutPolicy` 解决这件事：

| 字段 | 说明 |
| --- | --- |
| `run_timeout` | 单次执行的硬上限（秒），无论有没有进展都到点就打断 |
| `idle_timeout` | 单次执行"没有可见进展"的最长允许时间 |
| `refresh_on` | `idle_timeout` 由什么刷新：`"auto"`（任意回调/子调用）或 `"heartbeat"`（需要节点主动 `runtime.heartbeat()`） |

超时会抛 `NodeTimeoutError`（带 `node` / `kind` / `elapsed` / `run_timeout` 字段）。它**故意不继承 builtin `TimeoutError`**——因为 `TimeoutError` 是 `OSError` 子类，会被 `default_retry_on` 判成"代码 bug 不重试"；不继承才能享受"超时默认重试"。

一个硬性限制：**`timeout` 只能配在 async 节点上**。同步节点在 `compile()` 阶段就会被 `ValueError` 拒绝（同步 Python 没法安全地中断），所以工具/接口调用最好用 `async def` 节点。


In [8]:
import asyncio

from langgraph.errors import NodeTimeoutError
from langgraph.types import TimeoutPolicy

timeout_counter = {"n": 0}


async def slow_node(state: RetryState) -> RetryState:
    timeout_counter["n"] += 1
    await asyncio.sleep(0.5)      # 模拟一个慢接口
    return {"logs": ["终于返回了"]}


timeout_builder = StateGraph(RetryState)
timeout_builder.add_node(
    "slow_node",
    slow_node,                                        # 必须是 async 节点
    timeout=TimeoutPolicy(run_timeout=0.2),          # 单次最多 0.2s
    retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.001),   # 超时可被默认策略重试
)
timeout_builder.add_edge(START, "slow_node")
timeout_builder.add_edge("slow_node", END)
timeout_graph = timeout_builder.compile()

# Jupyter 里本来就跑着事件循环，所以直接用顶层 await（不要用 asyncio.run）
try:
    await timeout_graph.ainvoke({"logs": []})
except NodeTimeoutError as e:
    print("超时上抛：", type(e).__name__, e)
    print(f"  node={e.node} kind={e.kind} elapsed={round(e.elapsed, 3)} "
          f"run_timeout={e.run_timeout} 实际尝试次数={timeout_counter['n']}")

# 同步节点 + timeout：compile 阶段就拒绝
def sync_slow_node(state: RetryState) -> RetryState:
    time.sleep(0.5)
    return {"logs": ["sync 返回了"]}


try:
    bad_builder = StateGraph(RetryState)
    bad_builder.add_node("sync_slow_node", sync_slow_node, timeout=TimeoutPolicy(run_timeout=0.2))
    bad_builder.add_edge(START, "sync_slow_node")
    bad_builder.add_edge("sync_slow_node", END)
    bad_builder.compile()
except ValueError as e:
    print("同步节点配 timeout ->", type(e).__name__, ":", e)


超时上抛： NodeTimeoutError Node 'slow_node' exceeded its run timeout of 0.200s (elapsed: 0.201s).
  node=slow_node kind=run elapsed=0.201 run_timeout=0.2 实际尝试次数=3
同步节点配 timeout -> ValueError : Node timeouts are only supported for async nodes because sync Python execution cannot be safely cancelled in-process. Node 'sync_slow_node' is sync.


#### 8. 全图默认策略：`set_node_defaults`

逐个节点写 `retry_policy` 很快就会漏。`StateGraph.set_node_defaults()` 可以一次性给全图兜底：

```python
builder.set_node_defaults(
    retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.5),
    cache_policy=None,
    error_handler=None,
    timeout=None,
)
```

规则（来自 1.2.11 的实现）：

- 这些默认值在 **`compile()` 时**统一套用，**节点上自己写的同名参数优先级更高**。
- `retry_policy` / `timeout` 适用于**所有**节点，**包括 `error_handler` 节点**；而 `cache_policy` / `error_handler` 只作用于普通节点（缓存 handler 的结果不安全，handler 也不能自己触发自己）。
- **不会被子图继承**：把本图当节点嵌进父图时，子图要自己再设一遍。


In [9]:
fanout_counter = {"n": 0}


def always_503(state: RetryState) -> RetryState:
    fanout_counter["n"] += 1
    raise requests.HTTPError("503")


# 全图默认：每个节点最多试 4 次；START 扇出两个节点，它们会并发各自重试
defaults_builder = StateGraph(RetryState)
defaults_builder.set_node_defaults(retry_policy=RetryPolicy(max_attempts=4, initial_interval=0.001))
defaults_builder.add_node("branch_a", always_503)
defaults_builder.add_node("branch_b", always_503)
defaults_builder.add_edge(START, "branch_a")
defaults_builder.add_edge(START, "branch_b")
defaults_builder.add_edge("branch_a", END)
defaults_builder.add_edge("branch_b", END)
try:
    defaults_builder.compile().invoke({"logs": []})
except requests.HTTPError as e:
    print(f"两个并行节点各重试 4 次 = 实际执行 {fanout_counter['n']} 次 ->", type(e).__name__)

# 节点级配置覆盖图级默认值
fanout_counter["n"] = 0
override_builder = StateGraph(RetryState)
override_builder.set_node_defaults(retry_policy=RetryPolicy(max_attempts=4, initial_interval=0.001))
override_builder.add_node("branch_a", always_503,
                          retry_policy=RetryPolicy(max_attempts=1, initial_interval=0.001))  # 覆盖
override_builder.add_node("branch_b", always_503)
override_builder.add_edge(START, "branch_a")
override_builder.add_edge(START, "branch_b")
override_builder.add_edge("branch_a", END)
override_builder.add_edge("branch_b", END)
try:
    override_builder.compile().invoke({"logs": []})
except requests.HTTPError as e:
    print(f"branch_a 覆盖成 1 次、branch_b 仍 4 次 = 实际执行 {fanout_counter['n']} 次 ->", type(e).__name__)


两个并行节点各重试 4 次 = 实际执行 8 次 -> HTTPError


branch_a 覆盖成 1 次、branch_b 仍 4 次 = 实际执行 5 次 -> HTTPError


#### 9. Agent 场景：给 `ToolNode` 挂重试

把这一节和上一节的 Agent Loop 合起来看：`ToolNode` 本身也是一个节点，所以一样可以挂 `retry_policy`。这解决的是 Agent 里最高频的一类抖动——**工具本身是外部依赖**（查天气、检索、数据库），偶发失败时如果直接把异常文本塞进 `ToolMessage` 给模型，模型往往会顺着错误信息编答案；正确做法是让 LangGraph 在节点层重试几次，实在不行再用 `ToolNode(handle_tool_errors=...)` 兜成一条"工具不可用"的 `ToolMessage`。

下面手工构造一条带 `tool_calls` 的 `AIMessage`（不用调 LLM）来演示。这个例子同时把第 3 节的坑再踩一遍：工具里抛的是 `requests.ConnectionError`，**默认 `retry_on` 不重试它**，必须显式加进白名单。


In [10]:
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode

tool_counter = {"n": 0}


@tool(parse_docstring=True)
def flaky_weather(city: str) -> str:
    """查询指定城市的天气。

    Args:
        city: 城市名称，例如「北京」「上海」。

    Returns:
        该城市的天气描述字符串。
    """
    tool_counter["n"] += 1
    if tool_counter["n"] < 3:
        raise requests.ConnectionError("上游服务连接被重置")
    return f"{city}：晴，25°C"


# 手工造一条带 tool_calls 的 AIMessage，省掉一次 LLM 调用
ai_message = AIMessage(
    content="",
    tool_calls=[{"name": "flaky_weather", "args": {"city": "北京"}, "id": "call_1", "type": "tool_call"}],
)


def build_tool_graph(retry_policy):
    builder = StateGraph(MessagesState)
    builder.add_node("tools", ToolNode([flaky_weather]), retry_policy=retry_policy)
    builder.add_edge(START, "tools")
    builder.add_edge("tools", END)
    return builder.compile()


# 1) 用默认 retry_on：requests.ConnectionError 属于 OSError 系，不重试，一次就失败
tool_counter["n"] = 0
try:
    build_tool_graph(RetryPolicy(max_attempts=5, initial_interval=0.001)).invoke({"messages": [ai_message]})
except requests.ConnectionError as e:
    print(f"默认 retry_on：工具被调用 {tool_counter['n']} 次就失败了 ->", e)

# 2) 显式把 requests 的网络异常加进白名单：重试到第 3 次成功
tool_counter["n"] = 0
tool_graph = build_tool_graph(
    RetryPolicy(
        max_attempts=5,
        initial_interval=0.001,
        retry_on=[requests.ConnectionError, requests.ReadTimeout],   # 关键：显式声明
    )
)
result = tool_graph.invoke({"messages": [ai_message]})
print("显式 retry_on：工具被调用", tool_counter["n"], "次")
print("ToolMessage =", result["messages"][-1].content)


默认 retry_on：工具被调用 1 次就失败了 -> 上游服务连接被重置


显式 retry_on：工具被调用 3 次
ToolMessage = 北京：晴，25°C


#### 实践清单

1. **分清三类失败**：偶发外部故障 → 重试；卡死无响应 → 超时；持续性故障 → 降级兜底。三者要一起设计，不能只靠重试。
2. **重试次数 × 退避要能接受**：`max_attempts=5` + 默认退避 ≈ 7.5s 空转。整体链路上还有外层 HTTP 超时的话，把两者的关系算清楚，别让内层重试比外层超时还长。
3. **只重试幂等操作**。重试意味着副作用可能已经发生了一半：写库、发消息、扣款要么做成幂等（幂等键 / upsert），要么就别放在会被重试的节点里。
4. **`retry_on` 一定要显式写**。默认策略对 `requests` 的连接/超时类异常是"不重试"的（它们继承 `OSError`），而 `httpx` 的会重试——同一份业务代码换个 HTTP 库行为就变了。
5. **多套策略把兜底放最后并收窄 `retry_on`**，否则宽松的默认策略会把精细策略"抢答"掉。
6. **保留 jitter**。`jitter=False` 只适合本地调试；线上关掉 jitter 容易造成惊群。
7. **观测靠日志，不靠 debug 流**。`stream_mode="debug"` 一次节点执行只打一条记录，看不到重试；打开 `langgraph.pregel._retry` 这个 logger 才有 `Retrying task ... after N seconds (attempt M)`。
8. **重试不消耗 `recursion_limit` 步数**，两套预算各自独立；但"卡在重试里"依然会占用一次调用的墙钟时间，所以 Agent 整体仍需要 `recursion_limit` 和业务级步数预算兜底（见 `05-AgentLoop.ipynb`）。
9. **别用重试掩盖 bug**。`ValueError` / `TypeError` / `KeyError` 这类不重试是对的——把 `max_attempts` 调到 10 只会让错误晚 7 秒才暴露。
